## Successful Semantic Modelling for Power BI - "0. Create lab models"

Each attendee runs this ONCE in their OWN (blank) workspace, after the
bootstrap has imported the workshop notebooks. It:
1. Creates the attendee's own lakehouse
2. Adds OneLake shortcuts to the shared workshop_shared tables (no data copy)
3. Builds the CHECKPOINT lab models as Direct Lake over OneLake
(RLS, Scaling, DAX+Calendar, Triage, and the messy Baseline)

In Lab 1B attendees build their OWN clean star in the browser UX ("New semantic
model" from the LAKEHOUSE view = Direct Lake over OneLake) to learn the moves.
This script ALSO pre-builds "01 Star Schema (fixed)" as the reference / reveal /
fallback (and the model Lab 9 reuses), plus the later checkpoint models, so every
lab starts from a known-good, restartable point.

Pattern is adapted from Phil's proven dax-tips/DirectLakeWorkshop notebooks.
It is PARAMETERISED (resolves the shared location by NAME at runtime), so the
repo carries no tenant-specific GUIDs and is re-deliverable by anyone.

HOW TO RUN
Create this as a SPARK (PySpark) notebook in your own (blank) workspace, set
the config in Cell 2, then Run all. A Spark notebook is REQUIRED: its session
token carries the OneLake scope needed to create shortcuts (a pure Python
notebook token gets 403 InsufficientScopes), notebookutils is auto-injected,
and the TOM model edits (Cell 6+) are proven in Spark. semantic-link-labs is
installed in Cell 1. Spark start is ~1 min; each attendee runs this ONCE.

In [ ]:
# ---- CELL 1: INSTALL --------------------------------------------------------
# Alone, and first. In Fabric %pip restarts the Python interpreter, so anything
# defined before it is lost. Config and imports therefore live in Cell 2.
%pip install -q semantic-link-labs

In [ ]:
# ---- CELL 2: CONFIG + IMPORTS ----------------------------------------------
# The presenter shows these two on the welcome slide (names, not GUIDs).
SHARED_WORKSPACE = "Successful Semantic Modelling"   # where workshop_shared lives
SHARED_LAKEHOUSE = "workshop_shared"                 # the shared data lakehouse

# Your own lakehouse (created here if it does not exist).
MY_LAKEHOUSE = "workshop"

# Tables to shortcut from the shared lakehouse into yours.
SHARED_TABLES = ["Date", "Product", "Customer", "Territory", "Sales",
                 "UserAccess", "Sales_messy", "Product_messy"]

# Clean-star tables (the base most lab models use).
STAR_TABLES = ["Date", "Product", "Customer", "Territory", "Sales"]

# The Module 3 report is fetched rather than built here: "Test as role" will not
# run at all unless a report bound to the model already exists in the workspace.
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/dax-tips/SuccessfulSemanticModelling/main"
RLS_REPORT_BUNDLE = f"{GITHUB_RAW_BASE}/labs/reports/rls-report.parts.json"
RLS_REPORT_NAME = "03 RLS - Territory and Category"

# False = build ALL checkpoint models (the normal attendee path). True = build
# ONLY the clean star first, as a quick validation of the whole pattern.
VALIDATE_ONLY = False

# PRESENTER ONLY. Adds the Module 2 storage-mode pair: the same star built as
# Direct Lake over OneLake and over the SQL endpoint, so the difference can be
# read side by side in TMDL view. Attendees leave this False; Module 2 is a demo
# and four extra models would stretch the setup budget for the whole room.
BUILD_STORAGE_MODELS = False


import sempy
import sempy_labs as labs
from sempy import fabric
import notebookutils              # auto-injected in a Spark notebook; explicit import is harmless
import pandas as pd
import json, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# ---- CELL 3: CREATE YOUR LAKEHOUSE + RESOLVE IDS ---------------------------
existing = labs.list_lakehouses()["Lakehouse Name"]
if MY_LAKEHOUSE not in existing.values:
    fabric.create_lakehouse(MY_LAKEHOUSE)
    print(f"Created lakehouse '{MY_LAKEHOUSE}' - letting OneLake settle (10s)...")
    time.sleep(10)   # a freshly created item needs a moment before OneLake ops
else:
    print(f"Lakehouse '{MY_LAKEHOUSE}' already exists")

# Resolve ids, retrying briefly in case the item was only just created.
props = None
for _ in range(6):
    try:
        props = notebookutils.lakehouse.getWithProperties(MY_LAKEHOUSE)
        break
    except Exception as ex:
        print(f"  lakehouse not resolvable yet ({ex}); retrying in 5s...")
        time.sleep(5)
if props is None:
    raise RuntimeError(f"Lakehouse '{MY_LAKEHOUSE}' did not become resolvable - try re-running the cell")
workspace_id = props["workspaceId"]
lakehouse_id = props["id"]
workspace_name = sempy.fabric.resolve_workspace_name(workspace_id)
print(f"Workspace: {workspace_name} ({workspace_id})\nLakehouse: {MY_LAKEHOUSE} ({lakehouse_id})")

In [ ]:
# ---- CELL 4: SHORTCUT THE SHARED TABLES INTO YOUR LAKEHOUSE -----------------
# Self-diagnosing + adaptive. First READ the source lakehouse's REAL table paths
# (works for flat OR schema-enabled sources), then create each shortcut via the
# Fabric REST Shortcuts API pointing at that real path. Runs in a SPARK notebook
# so the token has OneLake scope. Per-table prints show exactly what happens and
# which tables (if any) are missing at the source.
from sempy_labs._helper_functions import resolve_workspace_name_and_id, resolve_lakehouse_name_and_id
(_, src_ws_id) = resolve_workspace_name_and_id(SHARED_WORKSPACE)
(_, src_lh_id) = resolve_lakehouse_name_and_id(lakehouse=SHARED_LAKEHOUSE, workspace=SHARED_WORKSPACE)
print(f"Source: {SHARED_LAKEHOUSE} ({src_lh_id}) in workspace {SHARED_WORKSPACE} ({src_ws_id})")

# Baked-in probe: what does the source actually expose, and at what path?
src_paths = {}
try:
    src_tbls = labs.lakehouse.get_lakehouse_tables(lakehouse=SHARED_LAKEHOUSE, workspace=SHARED_WORKSPACE)
    print(f"Source reports {len(src_tbls)} tables:")
    for _, r in src_tbls.iterrows():
        loc = str(r.get("Location", ""))
        rel = loc.split("/Tables/", 1)[1].rstrip("/") if "/Tables/" in loc else r["Table Name"]
        src_paths[r["Table Name"]] = rel
        print(f"   {r['Table Name']:<16} -> Tables/{rel}")
except Exception as ex:
    print(f"  WARN could not list source tables ({ex}); assuming flat Tables/<name>")

# Clear existing shortcuts in MY lakehouse (idempotent restart).
for _, row in labs.lakehouse.list_shortcuts(lakehouse=MY_LAKEHOUSE).iterrows():
    labs.lakehouse.delete_shortcut(shortcut_name=row["Shortcut Name"], lakehouse=MY_LAKEHOUSE)
    print(f"  cleared existing shortcut: {row['Shortcut Name']}")

# Create one shortcut per requested table via the sempy_labs WRAPPER. The wrapper
# calls the API with a properly-scoped Fabric client; the raw FabricRestClient
# gets 403 InsufficientScopes for shortcut writes. The generator's saveAsTable
# lowercased the source folders, so match case-INSENSITIVELY (table_name = the
# real source path) and pass shortcut_name = our Title case, so the attendee
# lakehouse presents clean 'Date','Sales',... to the Direct Lake model.
src_by_lower = {name.lower(): rel for name, rel in src_paths.items()}
made = 0
for t in SHARED_TABLES:
    rel = src_by_lower.get(t.lower(), t if not src_paths else None)
    if rel is None:
        print(f"  SKIP {t}: not present in source lakehouse '{SHARED_LAKEHOUSE}'")
        continue
    try:
        labs.lakehouse.create_shortcut_onelake(
            table_name=rel,
            source_lakehouse=SHARED_LAKEHOUSE, source_workspace=SHARED_WORKSPACE,
            destination_lakehouse=MY_LAKEHOUSE, shortcut_name=t)
        made += 1
        print(f"  shortcut created: {t:<16} -> Tables/{rel}")
    except Exception as e:
        print(f"  FAILED {t}: {str(e)[:300]}")
print(f"Shortcuts done. {made}/{len(SHARED_TABLES)} in place.")

In [ ]:
# ---- CELL 5: WAIT FOR SQL ENDPOINT, THEN METADATA REFRESH ------------------
# A brand-new lakehouse provisions its SQL Analytics Endpoint asynchronously.
# In a classroom we cannot wait for the ~15-min background job, so we (a) wait
# for the endpoint to report Success, then (b) force a metadata sync. BOTH loops
# are BOUNDED, so a fresh lakehouse (or a bad tenant day) can never spin forever.
def wait_for_sql_endpoint(max_wait_s=300, every_s=5):
    """Poll the lakehouse until its SQL endpoint is provisioned. Returns the
    endpoint id (or the last value seen if it times out)."""
    client = fabric.FabricRestClient()
    waited, sep_id = 0, None
    while waited <= max_wait_s:
        sep = (client.get(f"/v1/workspaces/{workspace_id}/lakehouses/{lakehouse_id}")
               .json().get("properties", {}).get("sqlEndpointProperties", {}))
        status, sep_id = sep.get("provisioningStatus"), sep.get("id")
        print(f"  SQL endpoint: status={status}  id={sep_id}  (waited {waited}s)")
        if status == "Success" and sep_id:
            return sep_id
        time.sleep(every_s); waited += every_s
    print("  WARNING: SQL endpoint not 'Success' within timeout - continuing anyway.")
    return sep_id

def triggerMetadataRefresh(max_polls=90, every_s=2):
    """Force a lakehouse -> SQL endpoint metadata sync. Bounded, visible polling;
    breaks on ANY terminal state (not just 'success') so it cannot hang."""
    client = fabric.FabricRestClient()
    sqlendpoint = (client.get(f"/v1/workspaces/{workspace_id}/lakehouses/{lakehouse_id}")
                   .json()["properties"]["sqlEndpointProperties"]["id"])
    uri = f"/v1.0/myorg/lhdatamarts/{sqlendpoint}"
    payload = {"commands": [{"$type": "MetadataRefreshExternalCommand"}]}
    batchId = client.post(uri, json=payload).json()["batchId"]
    statusuri = f"/v1.0/myorg/lhdatamarts/{sqlendpoint}/batches/{batchId}"
    terminal, state = ("success", "failed", "failure", "error", "cancelled", "completed"), None
    for _ in range(max_polls):
        state = str(client.get(statusuri).json().get("progressState", ""))
        if state.lower() in terminal:
            break
        time.sleep(every_s)
    print(f"Metadata refresh: {state}")
    return state

print("Waiting for SQL endpoint to provision (fresh lakehouse can take a couple of minutes)...")
wait_for_sql_endpoint()
triggerMetadataRefresh()

In [ ]:
# ---- CELL 6: HELPERS - create a Direct Lake OVER ONELAKE model -------------
def create_dl_model(model_name, tables, use_sql_endpoint=False):
    """Create a Direct Lake model over the given lakehouse tables. use_sql_endpoint
    False emits Direct Lake over OneLake, True routes through the SQL analytics
    endpoint - the Module 2 distinction."""
    flavour = "SQL endpoint" if use_sql_endpoint else "OneLake"
    print(f"  [{model_name}] generating Direct Lake over {flavour} for {len(tables)} tables: {tables}")
    ok = False
    for attempt in range(6):
        try:
            if fabric.list_items().query(
                    f"`Display Name`=='{model_name}' & Type=='SemanticModel'").shape[0] == 0:
                labs.directlake.generate_direct_lake_semantic_model(
                    dataset=model_name, tables=tables,
                    source=MY_LAKEHOUSE, source_type="Lakehouse",
                    source_workspace=workspace_name,
                    use_sql_endpoint=use_sql_endpoint,
                    workspace=workspace_name,
                    refresh=False)                   # existence guard above handles overwrite
            ok = True
            break
        except Exception as e:
            print(f"  model create hiccup (attempt {attempt + 1}/6): {e}")
            triggerMetadataRefresh()
            time.sleep(3)
    if not ok:
        raise RuntimeError(f"Could not create model '{model_name}' after 6 attempts")
    labs.refresh_semantic_model(dataset=model_name)   # frame onto the source
    print(f"  {model_name}: created as Direct Lake over {flavour}")


def build_star(model_name, tables=None, extra_setup=None, use_sql_endpoint=False):
    """Standard clean star: relationships, measures, date table, sort-by, hide.
    Optionally include extra tables and run an extra_setup(model_name) callback."""
    create_dl_model(model_name, tables or STAR_TABLES, use_sql_endpoint)

    print(f"  [{model_name}] adding relationships + measures + date table...")
    with labs.tom.connect_semantic_model(dataset=model_name, readonly=False) as tom:
        for r in list(tom.model.Relationships):
            tom.model.Relationships.Remove(r)
        for tbl in tom.model.Tables:            # idempotent: clear measures so re-runs don't clash
            for m in list(tbl.Measures):
                tbl.Measures.Remove(m)
        tom.add_relationship(from_table="Sales", from_column="OrderDateKey",
                             to_table="Date", to_column="DateKey",
                             from_cardinality="Many", to_cardinality="One")
        tom.add_relationship(from_table="Sales", from_column="ShipDateKey",
                             to_table="Date", to_column="DateKey",
                             from_cardinality="Many", to_cardinality="One",
                             is_active=False)   # role-playing (Module 6)
        tom.add_relationship(from_table="Sales", from_column="ProductKey",
                             to_table="Product", to_column="ProductKey",
                             from_cardinality="Many", to_cardinality="One")
        tom.add_relationship(from_table="Sales", from_column="CustomerKey",
                             to_table="Customer", to_column="CustomerKey",
                             from_cardinality="Many", to_cardinality="One")
        tom.add_relationship(from_table="Sales", from_column="TerritoryKey",
                             to_table="Territory", to_column="TerritoryKey",
                             from_cardinality="Many", to_cardinality="One")

        money = "\\$#,0.00;(\\$#,0.00);\\$#,0.00"
        tom.add_measure(table_name="Sales", measure_name="Total Sales",
                        expression="SUM(Sales[SalesAmount])", format_string=money)
        tom.add_measure(table_name="Sales", measure_name="Total Cost",
                        expression="SUM(Sales[Cost])", format_string=money)
        tom.add_measure(table_name="Sales", measure_name="Total Quantity",
                        expression="SUM(Sales[Quantity])", format_string="#,0")
        tom.add_measure(table_name="Sales", measure_name="Margin",
                        expression="[Total Sales] - [Total Cost]", format_string=money)
        tom.add_measure(table_name="Sales", measure_name="Order Count",
                        expression="DISTINCTCOUNT(Sales[OrderNumber])", format_string="#,0")

        tom.mark_as_date_table(table_name="Date", column_name="Date")

    # Sort By Columns (fixes MonthYear/MonthName/etc. sorting alphabetically).
    print(f"  [{model_name}] setting sort-by columns...")
    tw = labs.tom.TOMWrapper(dataset=model_name, workspace=workspace_name, readonly=False)
    tw.set_sort_by_column(table_name="Date", column_name="MonthName",   sort_by_column="Month")
    tw.set_sort_by_column(table_name="Date", column_name="MonthYear",   sort_by_column="MonthYearSort")
    tw.set_sort_by_column(table_name="Date", column_name="DayName",     sort_by_column="DayOfWeek")
    tw.set_sort_by_column(table_name="Date", column_name="QuarterName", sort_by_column="Quarter")
    tw.model.SaveChanges()

    # Hide keys + sort-helper columns so report authors use friendly fields.
    print(f"  [{model_name}] hiding keys + final reframe...")
    hide = {"Sales": ["SalesKey", "OrderDateKey", "ShipDateKey", "ProductKey",
                      "CustomerKey", "TerritoryKey"],
            "Date": ["DateKey", "MonthYearSort", "PriorYearDateKey", "PriorWeekDateKey"],
            "Product": ["ProductKey"], "Customer": ["CustomerKey"],
            "Territory": ["TerritoryKey"]}
    with labs.tom.connect_semantic_model(dataset=model_name, readonly=False) as tom:
        for tbl in tom.model.Tables:
            for c in tbl.Columns:
                if c.Name in hide.get(tbl.Name, []):
                    c.IsHidden = True
    labs.refresh_semantic_model(dataset=model_name)

    if extra_setup:
        extra_setup(model_name)
    print(f"  {model_name}: ready")

In [ ]:
# ---- CELL 7: CHECKPOINT-SPECIFIC ADD-ONS -----------------------------------
def add_slow_measures(model_name):
    """Module 7 triage: deliberately slow measures + expose a high-card column."""
    with labs.tom.connect_semantic_model(dataset=model_name, readonly=False) as tom:
        tom.add_measure(table_name="Sales", measure_name="Slow Sales (FE)",
                        expression="SUMX ( VALUES ( Sales[OrderNumber] ), CALCULATE ( SUM ( Sales[SalesAmount] ) ) )",
                        format_string="\\$#,0.00")
        tom.add_measure(table_name="Sales", measure_name="Slow Sales (SE)",
                        expression="SUMX ( Sales, Sales[Quantity] * Sales[UnitPrice] * ( 1 - Sales[Discount] ) )",
                        format_string="\\$#,0.00")
        for c in tom.model.Tables["Sales"].Columns:
            if c.Name == "OrderNumber":
                c.IsHidden = False


def link_user_access(model_name):
    """Module 3: bridge the RLS mapping table to Territory. UserAccess is on the many
    side, so the security filter has to travel many->one to reach the fact. The engine
    rejects SecurityFilterBehavior=BothDirections unless CrossFilterBehavior is too."""
    with labs.tom.connect_semantic_model(dataset=model_name, readonly=False) as tom:
        tom.add_relationship(from_table="UserAccess", from_column="TerritoryKey",
                             to_table="Territory", to_column="TerritoryKey",
                             from_cardinality="Many", to_cardinality="One",
                             cross_filtering_behavior="BothDirections",
                             security_filtering_behavior="BothDirections")
        for c in tom.model.Tables["UserAccess"].Columns:
            if c.Name == "TerritoryKey":
                c.IsHidden = True
    print(f"  [{model_name}] UserAccess -> Territory bridge added (cross-filter + security filter both directions)")


def build_baseline_messy(model_name):
    """Module 1 diagnose model with planted issues: text-key join, bidirectional
    relationship, two grains, dimension-with-fact-columns, and an unused Notes column."""
    create_dl_model(model_name, ["Sales_messy", "Product_messy", "Customer", "Date"])
    with labs.tom.connect_semantic_model(dataset=model_name, readonly=False) as tom:
        for r in list(tom.model.Relationships):
            tom.model.Relationships.Remove(r)
        for tbl in tom.model.Tables:            # idempotent: clear measures so re-runs don't clash
            for m in list(tbl.Measures):
                tbl.Measures.Remove(m)
        tom.add_relationship(from_table="Sales_messy", from_column="ProductName",
                             to_table="Product_messy", to_column="ProductName",
                             from_cardinality="Many", to_cardinality="One")            # text join
        tom.add_relationship(from_table="Sales_messy", from_column="CustomerKey",
                             to_table="Customer", to_column="CustomerKey",
                             from_cardinality="Many", to_cardinality="One",
                             cross_filtering_behavior="BothDirections")                # bidirectional
        tom.add_relationship(from_table="Sales_messy", from_column="OrderDateKey",
                             to_table="Date", to_column="DateKey",
                             from_cardinality="Many", to_cardinality="One")
        tom.add_measure(table_name="Sales_messy", measure_name="Total Sales",
                        expression="SUM(Sales_messy[SalesAmount])", format_string="\\$#,0.00")
    labs.refresh_semantic_model(dataset=model_name)
    print(f"  {model_name}: messy diagnose model ready")


def build_rls_report(model_name="03 RLS", report_name=RLS_REPORT_NAME):
    """Stamp the pre-built Module 3 report into this workspace, bound to the local model.

    Role testing renders through a report. Without one the service refuses with
    "To test row-level security, create a report with the semantic model and save it
    to the same workspace", which reads like a permissions problem and costs the room
    ten minutes. Shipping the report removes that from the lab entirely."""
    import base64
    import requests

    items = fabric.list_items()
    if not items[(items["Type"] == "Report") & (items["Display Name"] == report_name)].empty:
        print(f"  {report_name}: already here, left alone")
        return
    ds = items[(items["Type"] == "SemanticModel") & (items["Display Name"] == model_name)]
    if ds.empty:
        print(f"  {report_name}: SKIPPED, no '{model_name}' model in this workspace")
        return

    ws_id, ws_name = fabric.get_workspace_id(), fabric.resolve_workspace_name()
    parts = list(requests.get(RLS_REPORT_BUNDLE, timeout=120).json()["parts"])
    pbir = json.dumps({
        "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definitionProperties/2.0.0/schema.json",
        "version": "4.0",
        "datasetReference": {"byConnection": {"connectionString":
            f'Data Source="powerbi://api.powerbi.com/v1.0/myorg/{ws_name}";'
            f"initial catalog={model_name};integrated security=ClaimsToken;"
            f"semanticmodelid={ds.iloc[0]['Id']}"}},
    })
    parts.append({"path": "definition.pbir",
                  "payload": base64.b64encode(pbir.encode()).decode(),
                  "payloadType": "InlineBase64"})

    try:
        client = fabric.FabricRestClient()
        r = client.post(f"/v1/workspaces/{ws_id}/reports",
                        json={"displayName": report_name, "definition": {"parts": parts}})
        # 202 means accepted, not valid. The definition is checked afterwards, so a
        # broken report reports success here and then silently never appears.
        if r.status_code == 202:
            op = (r.headers.get("x-ms-operation-id")
                  or r.headers.get("Location", "").rsplit("/", 1)[-1])
            for _ in range(40):
                time.sleep(2)
                st = client.get(f"/v1/operations/{op}").json()
                if st.get("status") in ("Succeeded", "Failed"):
                    break
            if st.get("status") != "Succeeded":
                print(f"  {report_name}: FAILED - {st.get('error')}")
                return
        print(f"  {report_name}: ready (Module 3 needs this for Test as role)")
    except Exception as e:
        # Never let this sink the run; the models matter more and the report can
        # be hand-built in the lab if it comes to that.
        print(f"  {report_name}: FAILED - {e}")

In [ ]:
# ---- CELL 8: BUILD THE MODELS ----------------------------------------------
# "01 Star Schema (fixed)" is the reference attendees compare their Lab 1B build to,
# the presenter's debrief reveal, the fallback if someone stalls, and Lab 9's model.
print("Building '01 Star Schema (fixed)' (validates the whole pattern)...")
build_star("01 Star Schema (fixed)")
print("  -> open it, drop Total Sales by Date[MonthYear]; MonthYear should sort chronologically.")

if not VALIDATE_ONLY:
    print("Building the checkpoint models...")
    build_baseline_messy("01 Baseline (messy)")
    build_star("03 RLS", tables=STAR_TABLES + ["UserAccess"],
               extra_setup=link_user_access)                 # attendees add the roles
    build_rls_report()                                          # "Test as role" needs a report
    build_star("04 Scaling")                                    # aggregation is a lab / demo
    build_star("06 DAX + Calendar")                             # Calendar + calc group are a lab
    build_star("07 Slow Visual Triage", extra_setup=add_slow_measures)
    print("All checkpoint models built.")

    if BUILD_STORAGE_MODELS:
        print("Building the Module 2 storage-mode pair (presenter)...")
        build_star("02 Storage - Direct Lake (OneLake)", use_sql_endpoint=False)
        build_star("02 Storage - Direct Lake (SQL endpoint)", use_sql_endpoint=True)
        print("  -> open both in TMDL view and compare the partition mode / source lines.")
else:
    print("VALIDATE_ONLY is True - built only the clean star. Set it False to build the rest.")

In [ ]:
# ---- CELL 9: SUMMARY --------------------------------------------------------
print("\\nSemantic models in your workspace:")
display(fabric.list_items().query("Type=='SemanticModel'")[["Display Name", "Id"]])

In [ ]:
# ---- CELL 10: HAND THE SPARK SESSION BACK -----------------------------------
# This is the ONLY Spark notebook in the day; every other one is Python and starts
# in seconds. Without this the session sits idle for SESSION_TIMEOUT_MIN (30) after
# you finish, and with a full room that is fifty idle sessions holding capacity
# nobody is using - which is what makes the next person's run slow.
#
# NOTE: nothing runs after this. Keep it last.
mssparkutils.session.stop()